# Capstone B — Pass/Fail Classifier with a Defensible Threshold
#
**Dataset (โปรดอ่านก่อนใช้งาน):**
- *UCI SECOM* — McCann, M. & Johnston, A. (2008). *SECOM* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C54305 (CC BY 4.0)
- ไฟล์: `datasets/secom.zip` — 1,567 แถว × 590 features, label −1 = PASS / +1 = FAIL (104 fails ~6.6%)
- **เคสเมโทรโลยี:** threshold ของโมเดล ≡ **tolerance limit** — ตั้งต่ำไป = ตรวจซ้ำเปล่างานเยอะ, ตั้งสูงไป = ปล่อยของเสียออกโรงงาน
#
## เป้าหมายของแทร็กนี้
#
สร้าง classifier ที่ทนค่าหาย + คลาสไม่สมดุล, เลือก threshold จากต้นทุนจริง (ไม่ใช่ 0.5 กิ่ม ๆ), และป้องกันตัวเองด้วย pipeline ไม่ให้โกงโดยไม่รู้ตัว

In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise FileNotFoundError("ไม่พบ datasets/ — รันโน้ตบุ๊กจากใน repo")


DATA = data_dir()


## เตรียมข้อมูล (ให้มาแล้ว — รันผ่านก่อนทำข้อ)

In [2]:
import io  # noqa: E402
import zipfile  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import polars as pl  # noqa: E402

# ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)
plt.rcParams["font.family"] = [
    "DejaVu Sans",
    "Noto Sans Thai",
    "Leelawadee UI",
    "Tahoma",
    "Thonburi",
]

with zipfile.ZipFile(DATA / "secom.zip") as z:
    feats = pl.read_csv(
        io.BytesIO(z.read("secom.data")),
        separator=" ",
        has_header=False,
        new_columns=[f"feature_{i}" for i in range(1, 591)],
        schema_overrides=[pl.Float64] * 590,
        null_values="NaN",
    )
    lab = pl.read_csv(
        io.BytesIO(z.read("secom_labels.data")),
        separator=" ",
        quote_char='"',
        has_header=False,
        new_columns=["label", "timestamp"],
    )

secom = feats.with_columns(lab["label"].alias("label"))
names = secom.drop("label").columns
X = secom.drop("label").to_numpy()
y = (secom["label"] == 1).to_numpy().astype(int)  # 1 = FAIL
n = len(y)
cut = int(n * 0.8)
Xtr, Xte, ytr, yte = X[:cut], X[cut:], y[:cut], y[cut:]
print(f"{n} แถว | train FAIL {ytr.sum()} | test FAIL {yte.sum()}")

1567 แถว | train FAIL 87 | test FAIL 17


## ข้อ 1 — baseline ที่ต้องมีก่อนอวดโมเดล
#
วัด `DummyClassifier(strategy="most_frequent")` และโมเดล logistic + pipeline (impute → scale) ด้วย average precision บน TimeSeriesSplit — ใครชนะใครแพ้ รายงานพร้อม dispersion ข้าม fold
#
- **TODO:** `dummy = DummyClassifier(strategy="most_frequent")` — วัดด้วย `cross_val_score` เหมือนกัน
- **TODO:** `pipe_log = Pipeline([impute(median), scale, LogisticRegression(class_weight="balanced")])`
- พิมพ์ mean ± std ของ CV AP ทั้งสองตัว

## ข้อ 2 — XGBoost บน pipeline + จูนเบา ๆ
#
ลอง 2–3 ชุดค่า `max_depth` (2/3/5) และ `n_estimators` (100/300) ด้วย `scale_pos_weight` วัดด้วย CV เดียวกัน — เลือกชุดที่ชนะ **บน CV เท่านั้น** แล้วค่อยแตะ test ครั้งเดียว
#
- **TODO:** วนซ้อน depth × n_estimators บน `Pipeline([impute, XGBClassifier(scale_pos_weight=...)])`
- เก็บ mean CV AP เป็นตาราง — เลือกผู้ชนะก่อน แล้วค่อย fit เต็ม + วัด test AP

## ข้อ 3 — threshold = tolerance limit: จูนจากต้นทุน ไม่ใช่ 0.5
#
กำหนดต้นทุนสมมุติ: ปล่อยของเสียผ่าน (missed FAIL) เสียหาย 100 หน่วย/ชิ้น, ตรวจซ้ำของดี (false alarm) เสียหาย 5 หน่วย/ชิ้น — วาด **ต้นทุนรวมเทียบ threshold** (0.01–0.99) บน test หา threshold ที่ต้นทุนต่ำสุด แล้วตอบ: confusion matrix ที่ threshold นั้นเป็นอย่างไร และถ้าต้นทุน missed โตขึ้น 10 เท่า threshold เลื่อนทิศไหน
#
- **TODO:** `proba = model.predict_proba(Xte)[:,1]`
- วน `thr` ใน `np.linspace(0.01, 0.99, 99)`:
  - `pred = (proba >= thr)`
  - `cost = 100 * missed + 5 * false_alarms` (missed = `(pred==0)&(yte==1)`, false_alarm = `(pred == 1) & (yte == 0)`)
- พลอต cost ต่อ `thr`, รายงาน argmin + confusion matrix ณ จุดนั้น

## ข้อ 4 — อธิบายโมเดล (SHAP)
#
beeswarm + waterfall ของชิ้นที่โมเดลมั่นใจ FAIL สุดใน test — แล้วเขียน "แถลงการณ์ต่อผู้ตรวจ" หนึ่งประโยคด้วยชื่อ feature จริง
#
- **TODO:** `Zte = pd.DataFrame(imputer จาก pipeline .transform(Xte), columns=names)`
- `sv = shap.TreeExplainer(model)(Zte.iloc[:200])`
- beeswarm + waterfall ของ argmax `P(FAIL)`

## ข้อ 5 — สรุปส่งงาน
#
ย่อหน้า 3–5 บรรทัด: ตัวเลขที่รายงานได้กับผู้บริหารคืออะไร (ไม่ใช่ accuracy!), threshold ที่เลือกอยู่บนสมมุติฐานต้นทุนใด และต้อง validate สมมุติฐานนั้นกับใครก่อนใช้จริง